# SmolVLM-256M: Stop at EOS

This notebook tests how to stop the decode loop when the model ends the answer. The current script always runs all 64 decode steps. Each step costs about 8 ms, also after the answer ends.

The script compares three kinds of strategy on the same image and prompt:

- `fixed` runs all steps. This is the current behavior.
- `stopK` checks for the end token after every K steps and stops early.
- `capN` runs only N steps and does not check. This can cut a long answer.

A check needs a GPU-to-CPU sync, so a small K adds time at each check. A large K runs extra steps after the answer ends.

The script prints the steps run, the decode time, the total time of prefill and decode, the time saved against `fixed`, and whether the answer is complete.

In [21]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/eos_stop.py --image {REPO}/frame.jpg --max-len 1024

Loading weights: 100% 471/471 [00:00<00:00, 969.33it/s] 
strategy  steps  decode ms  total ms  saved ms  complete answer
answer tokens: 9
fixed        64      519.0     779.8       0.0  True
stop1         9       79.2     339.6     440.2  True
stop4        12      103.4     365.8     414.0  True
stop8        16      136.4     394.6     385.2  True
stop16       16      136.6     401.3     378.6  True
cap16        16      136.1     404.1     375.8  True
cap32        32      264.8     532.5     247.3  True


## Result

The test ran on a Colab GPU with one image. The answer has 9 tokens. All strategies give the complete answer.

| Strategy | Steps | Decode (ms) | Total (ms) | Saved (ms) | Complete answer |
|---|---|---|---|---|---|
| fixed | 64 | 519.0 | 779.8 | 0.0 | True |
| stop1 | 9 | 79.2 | 339.6 | 440.2 | True |
| stop4 | 12 | 103.4 | 365.8 | 414.0 | True |
| stop8 | 16 | 136.4 | 394.6 | 385.2 | True |
| stop16 | 16 | 136.6 | 401.3 | 378.6 | True |
| cap16 | 16 | 136.1 | 404.1 | 375.8 | True |
| cap32 | 32 | 264.8 | 532.5 | 247.3 | True |

Total is the prefill plus the decode.

- `stop1` is the best strategy. It cuts the decode from 519 ms to 79 ms. It cuts the total from 780 ms to 340 ms (56% less).
- The check costs almost nothing. A line through the runs gives about 8.0 ms for each step plus about 7 ms of fixed time. The line predicts 79.8 ms for `stop1`. The measured value is 79.2 ms.
- Each wasted step costs 8 ms, and a check costs less than 1 ms. Use K = 1.
- `cap16` and `cap32` give no gain over `stop1`. They can also cut a longer answer.
- Treat differences under about 10 ms in the total column as noise. The prefill part varies by about 5 ms.
- With `fast-gpu`, the request is about 378 ms with `stop1`. It is 823 ms with 64 steps (estimate from separate runs). Prefill is now about 75% of the request.
- The hybrid attention gain is now small. It is about 4 ms for a 9-token answer.
- The test used one image and one prompt. The saving changes with the answer length.
- `GraphDecoder.decode` and `generate` now use `stop1`. `bench.py` prints the time with and without the stop.